# Display features

In [2]:
import pickle
with open(r"C:\Users\bilel.guetarni\Desktop\ARTIX\data\artix.pkl", "rb") as f:
    patients = pickle.load(f)

In [10]:
print(patients[0].clinical.keys())
print(patients[0].id)
print(patients[0].clinical['SEX'])
print(patients[0].clinical['AGE'])
print(patients[0].clinical['ST_STAG'])
print(patients[0].clinical['ECOG'])

dict_keys(['STUDYID', 'USUBJID', 'CENTERID', 'ARMCD', 'ACTARMCD', 'INCDT', 'RANDT', 'SEX', 'AGE', 'BMI', 'ECOG', 'FUM', 'PA', 'OH', 'DIAB', 'SKIN', 'TTT', 'ATCD', 'P16', 'STAG', 'ST_STAG', 'ST_IMRT', 'ST_HPV', 'ST_CHEMO', 'HISTO', 'LAT', 'DIAM', 'NSTAG', 'GG', 'GG_SIZ', 'GG_NB', 'GG_FIX', 'LOC', 'ITT', 'MITT', 'PP', 'REPCPLT', 'REPDTC', 'BOREP1', 'ORR12', 'ORR24', 'DCD', 'DCDDT', 'DCDR', 'PROG', 'PROGDT', 'PROGLOC', 'PROGLR', 'PROGLRDT', 'CANCER2', 'CANCER2DT', 'CANCER2LOC', 'DDNDT', 'XERO12', 'XERO12_IMPUT1', 'XERO12_IMPUT2', 'TTSEQ', 'CHIMIO_TYP', 'CHIMIO_NB', 'CHIMIO_CUMUL', 'CHIMIO_CUMUL2', 'CHIMIOSTDT', 'CHIMIOENDDT', 'RTSTDT', 'RTENDT', 'RT_TYP', 'RT_NB', 'RT_INTER', 'COZ_INTER', 'SCAN_NB', 'SCAN_REPLDT', 'SCAN_REPLR', 'SCAN_REPLRP', 'SCAN_REPLSNUM', 'SCAN_REPLSDT'])
98
Male
49
Stage III
Completely ambulatory


In [9]:
import pandas
pandas.DataFrame(patients[0].clinical_measurements)

,type,value,visitID,sample
0,SSF,2820,Inclusion,NaN
1,SSF,1580,M6,NaN
2,SSF,1440,M12,NaN
3,SSF,1280,M18,NaN
4,SSF,980,M24,NaN
...,...,...,...,...
206,AE,0,M12,DYSPHAGIE
207,AE,0,M15,DYSPHAGIE
208,AE,Grade 1,M18,DYSPHAGIE
209,AE,0,M21,DYSPHAGIE


In [12]:
import os
path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\features\original\artix"
patient_folder = os.path.join(path_, [i for i in os.listdir(path_) if int(i) == int(patients[0].id)][0])
print(patient_folder)

C:\Users\bilel.guetarni\Desktop\ARTIX\features\original\artix\98


In [17]:
radiomics = pandas.read_csv(os.path.join(patient_folder, "deepnn.csv"))
radiomics["id"] = int(patients[0].id)
radiomics

,Unnamed: 0,oar,name,value,id
0,0,inferior_pharyngeal_constrictor,0,1.364924,98
1,1,inferior_pharyngeal_constrictor,1,1.500968,98
2,2,inferior_pharyngeal_constrictor,2,-0.960994,98
3,3,inferior_pharyngeal_constrictor,3,1.712207,98
4,4,inferior_pharyngeal_constrictor,4,-11.100934,98
...,...,...,...,...,...
4603,4603,oral_cavity,507,12.325427,98
4604,4604,oral_cavity,508,16.845053,98
4605,4605,oral_cavity,509,23.718340,98
4606,4606,oral_cavity,510,15.559124,98


In [ ]:
def load_features(path_, type_, patient_id, columns_to_keep=["oar", "name", "value"]):
    """
    load features

    args:
        path_ (str) path to features folder
        type_ (str) features type (radiomics, dosiomics, dvh, deepnn)
        patient_id (int,str) id of patient to find its folder
        columns_to_keep (list(str)) list of columns name to keep
    """
    patient_folder = os.path.join(path_, [i for i in os.listdir(path_) if int(i) == int(patient_id)][0])
    features = pandas.read_csv(os.path.join(patient_folder, f"{type_}.csv"))
    
    if "Unnamed: 0" in features.columns:
        features = features.drop(columns=["Unnamed: 0"])
    
    if columns_to_keep:
        features = features[columns_to_keep]
    
    return features

def load_radiomics(path_, patient_id, radiomics=["shape", "firstorder", "glcm", "gldm", "glrlm", "glszm", "ngtdm"]):
    df = load_features(path_, "radiomics", patient_id, columns_to_keep=None)
    df = df[(df["type"] == "original") & (df["class"].isin(radiomics))]
    return df.drop(columns=["type"])

def load_dosiomics(path_, patient_id, radiomics=["shape", "firstorder", "glcm", "gldm", "glrlm", "glszm", "ngtdm"]):
    df = load_features(path_, "dosiomics", patient_id, columns_to_keep=None)
    df = df[(df["type"] == "original") & (df["class"].isin(radiomics))]
    return df.drop(columns=["type"])

In [29]:
df = load_radiomics(path_, patients[0].id)
df

,oar,class,name,value,id
37,inferior_pharyngeal_constrictor,shape,Elongation,0.6332392723602007,98
38,inferior_pharyngeal_constrictor,shape,Flatness,0.17221602602169478,98
39,inferior_pharyngeal_constrictor,shape,LeastAxisLength,8.191254854755249,98
40,inferior_pharyngeal_constrictor,shape,MajorAxisLength,47.563836211871255,98
41,inferior_pharyngeal_constrictor,shape,Maximum2DDiameterColumn,44.41846462902562,98
...,...,...,...,...,...
1291,oral_cavity,ngtdm,Busyness,0.41364827812024374,98
1292,oral_cavity,ngtdm,Coarseness,0.00015605082009015784,98
1293,oral_cavity,ngtdm,Complexity,7699.555118661359,98
1294,oral_cavity,ngtdm,Contrast,0.002258607420755391,98


In [30]:
df = load_dosiomics(path_, patients[0].id)
df

,oar,class,name,value,id
37,inferior_pharyngeal_constrictor,shape,Elongation,0.6503859946301421,98
38,inferior_pharyngeal_constrictor,shape,Flatness,0.22645761555627789,98
39,inferior_pharyngeal_constrictor,shape,LeastAxisLength,10.951103391643514,98
40,inferior_pharyngeal_constrictor,shape,MajorAxisLength,48.35829152728146,98
41,inferior_pharyngeal_constrictor,shape,Maximum2DDiameterColumn,46.87216658103186,98
...,...,...,...,...,...
1291,oral_cavity,ngtdm,Busyness,135.46564239615049,98
1292,oral_cavity,ngtdm,Coarseness,0.0014106025927548165,98
1293,oral_cavity,ngtdm,Complexity,0.07725953792859573,98
1294,oral_cavity,ngtdm,Contrast,0.0042874171867065625,98


In [22]:
df = load_features(path_, "dvh", patients[0].id)
df

,oar,name,value,id
0,inferior_pharyngeal_constrictor,mean,50.965553505535084,98
1,inferior_pharyngeal_constrictor,min,40.59,98
2,inferior_pharyngeal_constrictor,max,62.55,98
3,inferior_pharyngeal_constrictor,D98,42.41 Gy,98
4,inferior_pharyngeal_constrictor,D10,58.70 Gy,98
...,...,...,...,...
247,oral_cavity,V55,0.00 %,98
248,oral_cavity,V60,0.00 %,98
249,oral_cavity,V65,0.00 %,98
250,oral_cavity,V70,0.00 %,98


In [23]:
df = load_features(path_, "deepnn", patients[0].id)
df

,oar,name,value,id
0,inferior_pharyngeal_constrictor,0,1.364924,98
1,inferior_pharyngeal_constrictor,1,1.500968,98
2,inferior_pharyngeal_constrictor,2,-0.960994,98
3,inferior_pharyngeal_constrictor,3,1.712207,98
4,inferior_pharyngeal_constrictor,4,-11.100934,98
...,...,...,...,...
4603,oral_cavity,507,12.325427,98
4604,oral_cavity,508,16.845053,98
4605,oral_cavity,509,23.718340,98
4606,oral_cavity,510,15.559124,98


# Build dataset

In [1]:
import pickle
with open(r"C:\Users\bilel.guetarni\Desktop\ARTIX\data\artix.pkl", "rb") as f:
    patients = pickle.load(f)

In [2]:
import pandas
import os

def load_features(path_, type_, patient_id, columns_to_keep=["oar", "name", "value"]):
    """
    load features

    args:
        path_ (str) path to features folder
        type_ (str) features type (radiomics, dosiomics, dvh, deepnn)
        patient_id (int,str) id of patient to find its folder
        columns_to_keep (list(str)) list of columns name to keep
    """
    patient_folder = os.path.join(path_, [i for i in os.listdir(path_) if int(i) == int(patient_id)][0])
    features = pandas.read_csv(os.path.join(patient_folder, f"{type_}.csv"))
    
    if "Unnamed: 0" in features.columns:
        features = features.drop(columns=["Unnamed: 0"])
    
    if columns_to_keep:
        features = features[columns_to_keep]
    
    return features

def load_radiomics(path_, patient_id, radiomics=["shape", "firstorder", "glcm", "gldm", "glrlm", "glszm", "ngtdm"]):
    df = load_features(path_, "radiomics", patient_id, columns_to_keep=None)
    df = df[(df["type"] == "original") & (df["class"].isin(radiomics))]
    return df.drop(columns=["type"])

def load_dosiomics(path_, patient_id, radiomics=["shape", "firstorder", "glcm", "gldm", "glrlm", "glszm", "ngtdm"]):
    df = load_features(path_, "dosiomics", patient_id, columns_to_keep=None)
    df = df[(df["type"] == "original") & (df["class"].isin(radiomics))]
    return df.drop(columns=["type"])

In [ ]:
import re

def transform_(x):
    if isinstance(x, str):
        digits = re.findall("\d", x)
        if len(digits) == 0:
            return None
        else:
            return int(digits[0])
    else:
        return None
    
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! adapt this for TCIA 
CLINICAL_MAPPING = {
    # SEX
    "Male": 1,
    "Female": 2,
    # "M": 1,
    # "F": 2,

    # CANCER STAGING
    "Stage III": 1,
    "Stage IVa": 2,
    "Stage IVb": 3,

    # ECOG
    "Asympatomatic": 0,
    "Completely ambulatory": 1,
    "lower than 50% in bed": 2,
}

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! adapt this for TCIA 
clinical = ['SEX', 'AGE', 'ST_STAG', 'ECOG']

features_path = r"C:\Users\bilel.guetarni\Desktop\ARTIX\features\original\artix"

df = []
for p in patients:
    patient_df = []

    for k in clinical:
        try:
            patient_df.append({"features": "clinical", "name": k, "value": int(p.clinical[k]), "id": int(p.id)})
        except ValueError:
            patient_df.append({"features": "clinical", "name": k, "value": CLINICAL_MAPPING[p.clinical[k]], "id": int(p.id)})

    # select dysphagia and xerostomia in CTCAE
    # transform values
    # change S0 into Inclusion
    # if patient has baseline tox, skip it
    # keep only M6 grade
    cm = pandas.DataFrame(p.clinical_measurements)
    cm = cm[(cm["type"] == "AE") & (cm["sample"].isin(["XEROSTOMIE", "DYSPHAGIE"]))]
    cm["value"] = cm["value"].apply(transform_)
    cm.loc[(cm["visitID"] == "S0"), "visitID"] = "Inclusion"
    if (cm[(cm["visitID"] == "Inclusion") & (cm["sample"] == "XEROSTOMIE")]["value"].astype('Int64') > 0).any():
        continue
    cm = cm[cm["visitID"] == "M6"]

    # transform tox value to binary
    cm.loc[(cm["value"] < 2), "value"] = 0
    cm.loc[(cm["value"] >= 2), "value"] = 1

    for _, row in cm.iterrows():
        patient_df.append({"features": "clinical", "name": row["sample"], "value": row["value"], "id": int(p.id)})

    try:
        # radiomics
        radiomics = load_radiomics(features_path, p.id)
        radiomics["id"] = int(p.id)
        radiomics["features"] = "radiomics"
        patient_df.extend(radiomics.to_dict(orient="records"))
    except (FileNotFoundError, IndexError):
        pass

    try:
        # dosiomics
        dosiomics = load_dosiomics(features_path, p.id)
        dosiomics["id"] = int(p.id)
        dosiomics["features"] = "dosiomics"
        patient_df.extend(dosiomics.to_dict(orient="records"))
    except (FileNotFoundError, IndexError):
        pass

    try:
        # dvh
        dvh = load_features(features_path, "dvh", p.id)
        dvh["id"] = int(p.id)
        dvh["features"] = "dvh"
        patient_df.extend(dvh.to_dict(orient="records"))
    except (FileNotFoundError, IndexError):
        pass

    try:
        # deepnn
        deepnn = load_features(features_path, "deepnn", p.id)
        deepnn["id"] = int(p.id)
        deepnn["features"] = "deepnn"
        patient_df.extend(deepnn.to_dict(orient="records"))
    except (FileNotFoundError, IndexError):
        pass

    df.extend(patient_df)

In [3]:
combined_path = r"C:\Users\bilel.guetarni\Desktop\tmp\artix_original.csv"
# pandas.DataFrame(df).to_csv(combined_path, index=False)

# Fit

In [4]:
import pandas
df = pandas.read_csv(combined_path)

In [5]:
# filter patients without xerostomia label
subdf = df[df["name"] == "XEROSTOMIE"]
subdf = subdf[pandas.isna(subdf["value"])]
patients_to_exclude = list(subdf["id"].unique())

# add patients without any XEROSTOMIA line
subdf = df[df["name"] == "XEROSTOMIE"]
patients_to_exclude.extend(list(set(set(df["id"].unique()).difference(subdf["id"].unique()))))

print("patients excluded due to absent xerostomia label:")
print(patients_to_exclude)

df = df[~df["id"].isin(patients_to_exclude)]

patients excluded due to absent xerostomia label:
[63, 59, 44, 29, 128, 107, 1, 25, 53, 113, 123]


FEATURES = {
    "clinical": ["SEX", "AGE", "ST_STAG", "ECOG"],
    "radiomics": ["shape", "firstorder", "glcm", "gldm", "glrlm", "glszm", "ngtdm"],
    "dosiomics": ["shape", "firstorder", "glcm", "gldm", "glrlm", "glszm", "ngtdm"],
    "dvh": ["mean", "min", "max", 
            "D98", "D10", "D20", "D30", "D40", "D50", "D60", "D70", "D80", "D90", 
            "V5", "V10", "V15", "V20", "V25", "V30", "V35", "V40", "V45", "V50", "V55", "V60", "V65", "V70", "V75"],
    "deepnn": "all",
}

In [6]:
list_of_features = df.groupby(["features", "class", "name"], dropna=False).size().reset_index().rename(columns={0:'count'})

missing_class_rows = filter(lambda row: pandas.isna(row[1]["class"]), list_of_features.iterrows())

with_class_rows = filter(lambda row: not(pandas.isna(row[1]["class"])), list_of_features.iterrows())

list_of_features = [
    *[f"{row['features']}_{row['name']}" for _, row in missing_class_rows],
    *[f"{row['features']}_{row['class']}_{row['name']}" for _, row in with_class_rows],
]

print(len(list_of_features))

760


In [44]:
list_of_features

['clinical_AGE',
 'clinical_DYSPHAGIE',
 'clinical_ECOG',
 'clinical_SEX',
 'clinical_ST_STAG',
 'clinical_XEROSTOMIE',
 'deepnn_0',
 'deepnn_1',
 'deepnn_10',
 'deepnn_100',
 'deepnn_101',
 'deepnn_102',
 'deepnn_103',
 'deepnn_104',
 'deepnn_105',
 'deepnn_106',
 'deepnn_107',
 'deepnn_108',
 'deepnn_109',
 'deepnn_11',
 'deepnn_110',
 'deepnn_111',
 'deepnn_112',
 'deepnn_113',
 'deepnn_114',
 'deepnn_115',
 'deepnn_116',
 'deepnn_117',
 'deepnn_118',
 'deepnn_119',
 'deepnn_12',
 'deepnn_120',
 'deepnn_121',
 'deepnn_122',
 'deepnn_123',
 'deepnn_124',
 'deepnn_125',
 'deepnn_126',
 'deepnn_127',
 'deepnn_128',
 'deepnn_129',
 'deepnn_13',
 'deepnn_130',
 'deepnn_131',
 'deepnn_132',
 'deepnn_133',
 'deepnn_134',
 'deepnn_135',
 'deepnn_136',
 'deepnn_137',
 'deepnn_138',
 'deepnn_139',
 'deepnn_14',
 'deepnn_140',
 'deepnn_141',
 'deepnn_142',
 'deepnn_143',
 'deepnn_144',
 'deepnn_145',
 'deepnn_146',
 'deepnn_147',
 'deepnn_148',
 'deepnn_149',
 'deepnn_15',
 'deepnn_150',
 'dee

In [ ]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, balanced_accuracy_score, precision_score, recall_score, f1_score, log_loss
from sklearn.decomposition import PCA

def display_split_stats(split):
    stats = {j: list(split).count(j) for j in set(split)}
    for j, c in stats.items():
        print(f"\t {j}: {c} ({int(100*c/len(split))}%)")

def cross_validation(X, Y, kfold=3, bootstrap=None, feature_selection=None, feature_reduction_N=None):
    """
    Perform cross validation training and return classification metrics as dict

    args
        X (numpy.ndarray) training data input of shape (n_samples, n_features)
        Y (numpy.ndarray) training data expected output of shape (n_samples)
        kfold (int) number of folds
        bootstrap (int) number of time to eprform the training with random splitting, if None apply k-fold
        feature_selection (str) feature selection method to apply (see sklearn)
        feature_reduction_N (int) number of dimension to reduce data into using PCA
    """

    metrics = []
    reductor = []
    if bootstrap:
        raise NotImplementedError() #TODO
    else:
        kf = KFold(n_splits=kfold)
        for i, (train_index, test_index) in enumerate(kf.split(X)):
            x_train, y_train = X[train_index], Y[train_index]
            x_test, y_test = X[test_index], Y[test_index]

            # print("fold ", i+1)
            # print("train:")
            # display_split_stats(y_train)
            # print("test:")
            # display_split_stats(y_test)

            if feature_reduction_N:
                print(f"transforming input from {x_train.shape[1]} to {feature_reduction_N} dimensions..")
                reduc = PCA(n_components=feature_reduction_N, random_state=0)
                reduc.fit(x_train)
                x_train = reduc.transform(x_train)
                x_test = reduc.transform(x_test)
                reductor.append(reduc)

            clf = LogisticRegression(random_state=0, verbose=0)
            clf.fit(x_train, y_train)
            y_pred = clf.predict(x_test)
            y_pred_proba = clf.predict_proba(x_test)
            metrics.append({
                "acc": accuracy_score(y_test, y_pred),
                "auc": roc_auc_score(y_test, y_pred),
                "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred, zero_division=0),
                "recall": recall_score(y_test, y_pred, zero_division=0),
                "f1_score": f1_score(y_test, y_pred, zero_division=0),
                "log_loss": log_loss(y_test, y_pred_proba),
            })
    return metrics, reductor

## 001-001

In [8]:
radiomics_df = df[df["features"].isin(["radiomics", "clinical"])]
print(radiomics_df["class"].unique())
print(radiomics_df["name"].unique())
print(radiomics_df["oar"].unique())

[nan 'shape' 'firstorder' 'glcm' 'gldm' 'glrlm' 'glszm' 'ngtdm']
['SEX' 'AGE' 'ST_STAG' 'ECOG' 'XEROSTOMIE' 'DYSPHAGIE' 'Elongation'
 'Flatness' 'LeastAxisLength' 'MajorAxisLength' 'Maximum2DDiameterColumn'
 'Maximum2DDiameterRow' 'Maximum2DDiameterSlice' 'Maximum3DDiameter'
 'MeshVolume' 'MinorAxisLength' 'Sphericity' 'SurfaceArea'
 'SurfaceVolumeRatio' 'VoxelVolume' '10Percentile' '90Percentile' 'Energy'
 'Entropy' 'InterquartileRange' 'Kurtosis' 'Maximum'
 'MeanAbsoluteDeviation' 'Mean' 'Median' 'Minimum' 'Range'
 'RobustMeanAbsoluteDeviation' 'RootMeanSquared' 'Skewness' 'TotalEnergy'
 'Uniformity' 'Variance' 'Autocorrelation' 'ClusterProminence'
 'ClusterShade' 'ClusterTendency' 'Contrast' 'Correlation'
 'DifferenceAverage' 'DifferenceEntropy' 'DifferenceVariance' 'Id' 'Idm'
 'Idmn' 'Idn' 'Imc1' 'Imc2' 'InverseVariance' 'JointAverage' 'JointEnergy'
 'JointEntropy' 'MCC' 'MaximumProbability' 'SumAverage' 'SumEntropy'
 'SumSquares' 'DependenceEntropy' 'DependenceNonUniformity'
 'Dep

In [ ]:
import os, json
import numpy as np
import tqdm
from sklearn.preprocessing import normalize, scale, minmax_scale
import plotly.express as px

def pca_visualization(x, y):
    x = PCA(n_components=2, random_state=0).fit_transform(x)
    fig = px.scatter(x=x[:,0], y=x[:,1], color=y)
    fig.show()

def get_unique_combos(df, col1, col2):
    unique_combos = df[[col1, col2]].drop_duplicates()
    return list(unique_combos.itertuples(index=False, name=None))

exp_code = "001"
exp = [
    # parotid ipsi
    {"feature_reduction_N": None, "class": ["shape"], "oar": "parotid_gland_ipsi"},
    {"feature_reduction_N": None, "class": ["firstorder"], "oar": "parotid_gland_ipsi"},
    {"feature_reduction_N": None, "class": ['glcm', 'gldm', 'glrlm', 'glszm', 'ngtdm'], "oar": "parotid_gland_ipsi"},
    {"feature_reduction_N": None, "class": [], "oar": "parotid_gland_ipsi"}, # all classes

    # parotid contra
    {"feature_reduction_N": None, "class": ["shape"], "oar": "parotid_gland_contra"},
    {"feature_reduction_N": None, "class": ["firstorder"], "oar": "parotid_gland_contra"},
    {"feature_reduction_N": None, "class": ['glcm', 'gldm', 'glrlm', 'glszm', 'ngtdm'], "oar": "parotid_gland_contra"},
    {"feature_reduction_N": None, "class": [], "oar": "parotid_gland_contra"},
]

for exp_params in tqdm.tqdm(exp):
    # create experience name
    if len(exp_params["class"]) == 0:
        features_names = "all"
    else:
        features_names = "_".join(exp_params["class"])
    exp_name = f"{exp_code}_{exp_params['oar']}_{features_names}"

    # select data containing OARs
    exp_df = radiomics_df[(radiomics_df["oar"] == exp_params["oar"])]
    if len(exp_params["class"]) > 0:
        exp_df = exp_df[exp_df["class"].isin(exp_params["class"])]
    
    # build X,Y
    patients = exp_df["id"].unique()
    features = get_unique_combos(exp_df, "class", "name")
    X = [[exp_df[(exp_df["id"] == int(p)) & (exp_df["class"] == class_) & (exp_df["name"] == name_)]["value"].item() for class_, name_ in features] for p in patients]
    Y = [df[(df["id"] == int(p)) & (df["name"] == "XEROSTOMIE")]["value"].item() for p in patients]
    X = np.array(X, dtype=np.float32)
    Y = np.array(Y, dtype=np.float16).astype(dtype=np.int16)

    # normalize features
    # X = scale(X, axis=0)
    X = minmax_scale(X, axis=0)

    # print("X: ", X.shape)
    # print("Y: ", Y.shape)

    # pca_visualization(X, Y)

    # fit model
    metrics, _ = cross_validation(X, Y, kfold=3, feature_reduction_N=exp_params["feature_reduction_N"])

    # save results
    out_dir = os.path.join("./experiments", exp_code, exp_name)
    os.makedirs(out_dir, exist_ok=True)
    pandas.DataFrame(metrics).to_csv(os.path.join(out_dir, "metrics.csv"))

    # save exp params
    with open(os.path.join(out_dir, "params.json"), "w") as f:
        json.dump(exp_params, f)

  0%|          | 0/8 [00:00<?, ?it/s]

 12%|█▎        | 1/8 [00:01<00:11,  1.71s/it]

X:  (46, 14)
Y:  (46,)
fold  1
train:
	 0: 22 (73%)
	 1: 8 (26%)
test:
	 0: 11 (68%)
	 1: 5 (31%)
fold  2
train:
	 0: 23 (74%)
	 1: 8 (25%)
test:
	 0: 10 (66%)
	 1: 5 (33%)
fold  3
train:
	 0: 21 (67%)
	 1: 10 (32%)
test:
	 0: 12 (80%)
	 1: 3 (20%)


 25%|██▌       | 2/8 [00:03<00:10,  1.76s/it]

X:  (46, 18)
Y:  (46,)
fold  1
train:
	 0: 22 (73%)
	 1: 8 (26%)
test:
	 0: 11 (68%)
	 1: 5 (31%)
fold  2
train:
	 0: 23 (74%)
	 1: 8 (25%)
test:
	 0: 10 (66%)
	 1: 5 (33%)
fold  3
train:
	 0: 21 (67%)
	 1: 10 (32%)
test:
	 0: 12 (80%)
	 1: 3 (20%)


...

X:  (46, 18)
Y:  (46,)
fold  1
train:
	 0: 22 (73%)
	 1: 8 (26%)
test:
	 0: 11 (68%)
	 1: 5 (31%)
fold  2
train:
	 0: 23 (74%)
	 1: 8 (25%)
test:
	 0: 10 (66%)
	 1: 5 (33%)
fold  3
train:
	 0: 21 (67%)
	 1: 10 (32%)
test:
	 0: 12 (80%)
	 1: 3 (20%)


 88%|████████▊ | 7/8 [00:21<00:03,  3.17s/it]

X:  (46, 75)
Y:  (46,)
fold  1
train:
	 0: 22 (73%)
	 1: 8 (26%)
test:
	 0: 11 (68%)
	 1: 5 (31%)
fold  2
train:
	 0: 23 (74%)
	 1: 8 (25%)
test:
	 0: 10 (66%)
	 1: 5 (33%)
fold  3
train:
	 0: 21 (67%)
	 1: 10 (32%)
test:
	 0: 12 (80%)
	 1: 3 (20%)


100%|██████████| 8/8 [00:28<00:00,  3.53s/it]

X:  (46, 107)
Y:  (46,)
fold  1
train:
	 0: 22 (73%)
	 1: 8 (26%)
test:
	 0: 11 (68%)
	 1: 5 (31%)
fold  2
train:
	 0: 23 (74%)
	 1: 8 (25%)
test:
	 0: 10 (66%)
	 1: 5 (33%)
fold  3
train:
	 0: 21 (67%)
	 1: 10 (32%)
test:
	 0: 12 (80%)
	 1: 3 (20%)


# Analysis

In [ ]:
import os
import pandas
import plotly.express as px

path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\001"

result_df = []
for exp in os.listdir(path_):
    metrics_df = pandas.read_csv(os.path.join(path_, exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"exp": exp, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

print(len(result_df))
result_df = pandas.DataFrame(result_df)

for metric in result_df["metric"].unique():
    metric_df = result_df[result_df["metric"] == metric]
    fig = px.bar(metric_df, x="exp", y="value", title=metric)
    fig.show()

56
